# Exploratory Data Analysis

Perform exploratory data analysis and data cleaning in order to produce a cleaned CSV and store it in Weights & Biases.

In [1]:
import os
import importlib
import tempfile

import pandas as pd
import wandb

importlib.import_module('src.income-prediction'); # fix Artifact.file() method

In [2]:
wandb_project = os.getenv('WANDB_PROJECT', 'income-prediction')
wandb_group = os.getenv('WANDB_RUN_GROUP', None)

## Weights & Biases Run

In [3]:
run = wandb.init(project=wandb_project, group=wandb_group, job_type='eda')

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\A6SC2D5\_netrc.
wandb: Currently logged in as: cariad-robert-abel (cariad-robert-abel-cariad-se) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


## Load Latest Raw

Load latest uncleaned (raw) data and look at it. We uploaded the raw CSV like `wandb artifact put ...` (but with metadata) beforehand.

In [4]:
artifact = run.use_artifact('census-income:latest', type='raw-data').file()

In [5]:
df = pd.read_csv(artifact, low_memory=False)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 32561 entries, 0 to 32560
Data columns (total 15 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   age              32561 non-null  int64
 1    workclass       32561 non-null  str  
 2    fnlgt           32561 non-null  int64
 3    education       32561 non-null  str  
 4    education-num   32561 non-null  int64
 5    marital-status  32561 non-null  str  
 6    occupation      32561 non-null  str  
 7    relationship    32561 non-null  str  
 8    race            32561 non-null  str  
 9    sex             32561 non-null  str  
 10   capital-gain    32561 non-null  int64
 11   capital-loss    32561 non-null  int64
 12   hours-per-week  32561 non-null  int64
 13   native-country  32561 non-null  str  
 14   salary          32561 non-null  str  
dtypes: int64(6), str(9)
memory usage: 3.7 MB


In [6]:
df.head(n=5)

,age,workclass,fnlgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,salary
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


### Strip Whitespace

The instructions specify that we may need to clean the data:

- This data is messy, try to open it in pandas and see what you get.
- To clean it, use your favorite text editor to remove all spaces.

So we'll strip whitespace first (programmatically, not manually) and then analyze further.

In [7]:
# get only string columns
def _find_leading_trailing_whitespace(df: pd.DataFrame) -> pd.DataFrame:
    # use vectorized string accessor methods: https://pandas.pydata.org/docs/user_guide/text.html
    return df.select_dtypes(include='string').apply(lambda col: col.str.contains(r'^\s|\s$', regex=True))

_find_leading_trailing_whitespace(df).sum()

workclass         32561
education         32561
marital-status    32561
occupation        32561
relationship      32561
race              32561
sex               32561
native-country    32561
salary            32561
dtype: int64

Apparently, there are initial spaces in column names as well as column values. Try to load the data differently first and check results.

In [8]:
# luckily, pandas can already skip whitespace *after* delimiter: https://pandas.pydata.org/docs/user_guide/io.html
df = pd.read_csv(artifact, skipinitialspace=True, low_memory=False)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 32561 entries, 0 to 32560
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   age             32561 non-null  int64
 1   workclass       32561 non-null  str  
 2   fnlgt           32561 non-null  int64
 3   education       32561 non-null  str  
 4   education-num   32561 non-null  int64
 5   marital-status  32561 non-null  str  
 6   occupation      32561 non-null  str  
 7   relationship    32561 non-null  str  
 8   race            32561 non-null  str  
 9   sex             32561 non-null  str  
 10  capital-gain    32561 non-null  int64
 11  capital-loss    32561 non-null  int64
 12  hours-per-week  32561 non-null  int64
 13  native-country  32561 non-null  str  
 14  salary          32561 non-null  str  
dtypes: int64(6), str(9)
memory usage: 3.7 MB


In [9]:
_find_leading_trailing_whitespace(df).sum()

workclass         0
education         0
marital-status    0
occupation        0
relationship      0
race              0
sex               0
native-country    0
salary            0
dtype: int64

This seems to have worked quite well. Let's print unique values per text column.

In [10]:
pd.concat(
    {col: pd.Series(sorted(values.unique())) for col, values in df.select_dtypes(include='string').items()},
    axis=1
).fillna('')

,workclass,education,marital-status,occupation,relationship,race,sex,native-country,salary
0,?,10th,Divorced,?,Husband,Amer-Indian-Eskimo,Female,?,<=50K
1,Federal-gov,11th,Married-AF-spouse,Adm-clerical,Not-in-family,Asian-Pac-Islander,Male,Cambodia,>50K
2,Local-gov,12th,Married-civ-spouse,Armed-Forces,Other-relative,Black,,Canada,
3,Never-worked,1st-4th,Married-spouse-absent,Craft-repair,Own-child,Other,,China,
4,Private,5th-6th,Never-married,Exec-managerial,Unmarried,White,,Columbia,
5,Self-emp-inc,7th-8th,Separated,Farming-fishing,Wife,,,Cuba,
6,Self-emp-not-inc,9th,Widowed,Handlers-cleaners,,,,Dominican-Republic,
7,State-gov,Assoc-acdm,,Machine-op-inspct,,,,Ecuador,
8,Without-pay,Assoc-voc,,Other-service,,,,El-Salvador,
9,,Bachelors,,Priv-house-serv,,,,England,


### Replace Question Marks

Let's replace the question marks with proper `NaN` so they don't end up as strings in out data.

In [11]:
row_before = df[(df == '?').any(axis=1)].iloc[0].copy()

In [12]:
df.replace('?', pd.NA, inplace=True);

In [13]:
row_before

age                               40
workclass                    Private
fnlgt                         121772
education                  Assoc-voc
education-num                     11
marital-status    Married-civ-spouse
occupation              Craft-repair
relationship                 Husband
race              Asian-Pac-Islander
sex                             Male
capital-gain                       0
capital-loss                       0
hours-per-week                    40
native-country                     ?
salary                          >50K
Name: 14, dtype: object

In [14]:
df.iloc[row_before.name]

age                               40
workclass                    Private
fnlgt                         121772
education                  Assoc-voc
education-num                     11
marital-status    Married-civ-spouse
occupation              Craft-repair
relationship                 Husband
race              Asian-Pac-Islander
sex                             Male
capital-gain                       0
capital-loss                       0
hours-per-week                    40
native-country                   NaN
salary                          >50K
Name: 14, dtype: object

### Rename Final Weight Column

The `fnlgt` column is actually the `AFNLWGT` column, which standas for "Final Weight" according to the Census Bureau's naming scheme. This value indicates how many people the specific row of the census data actually represents. Therefore, let's rename the column to its proper name. We need to take care to *not* use this as input for our training algorithm, because this doesn't represent a feature at all!

**UPDATE**: I was wrong: `fnlwgt` is actually a feature and does not represent the number of people represented by a specific row in the census data.
From the original description in the [Adult Dataset](https://archive.ics.uci.edu/dataset/2/adult):

> Description of fnlwgt (final weight)
> 
> The weights on the CPS files are controlled to independent estimates of the
> civilian noninstitutional population of the US.  These are prepared monthly
> for us by Population Division here at the Census Bureau.  We use 3 sets of
> controls.
>  These are:
>          1.  A single cell estimate of the population 16+ for each state.
>          2.  Controls for Hispanic Origin by age and sex.
>          3.  Controls by Race, age and sex.
> 
> We use all three sets of controls in our weighting program and "rake" through
> them 6 times so that by the end we come back to all the controls we used.
> 
> The term estimate refers to population totals derived from CPS by creating
> "weighted tallies" of any specified socio-economic characteristics of the
> population.
> 
> People with similar demographic characteristics should have
> similar weights.  There is one important caveat to remember
> about this statement.  That is that since the CPS sample is
> actually a collection of 51 state samples, each with its own
> probability of selection, the statement only applies within
> state.

This means it *is* a feature, but probably the data churning by the census bureau adds interesting qualities.
However, since the information which state collected which sample row is lost, this might not be as meaningful as we'd like!

In [15]:
df.columns = df.columns.str.replace('fnlgt', 'fnlwgt')
df.head(n=3)

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,salary
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K


### Drop `education-num` Column

The `education` and `education-num` columns match as latter is simply the integer enumeration of the string values. However, they don't actually form a quantitative feature, as it's not continous.

In [16]:
del df['education-num']
df.head(n=3)

,age,workclass,fnlwgt,education,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,salary
0,39,State-gov,77516,Bachelors,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K


### Finish Weights & Biases Run

Finish run and upload data.

In [17]:
with tempfile.TemporaryDirectory(prefix='income-') as tmpdir:
    filename = os.path.join(tmpdir, 'cleaned-data.csv')
    df.to_csv(filename, index=False)

    cleaned = wandb.Artifact('census-income-clean', type='cleaned-data', description='Cleaned UCI Census Income (Adult) Dataset')
    cleaned.add_file(filename)
    run.log_artifact(cleaned)
    cleaned.wait()

run.finish()